# Weight Statistics for `history_3d_residual_cnn_1`

Этот ноутбук читает `checkpoint_epoch_*.pt`, считает статистики весов по эпохам и строит графики вместе с `epoch_metrics.csv`.

In [1]:
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
ANALYZE_DIR = Path('.').resolve()
REPO_DIR = ANALYZE_DIR.parent
RUN_DIR = REPO_DIR / 'output' / 'history_3d_residual_cnn_1'
METRICS_PATH = RUN_DIR / 'epoch_metrics.csv'

checkpoint_paths = sorted(
    RUN_DIR.glob('checkpoint_epoch_*.pt'),
    key=lambda path: int(re.search(r'checkpoint_epoch_(\d+)\.pt$', path.name).group(1))
)

print(f'Run dir: {RUN_DIR}')
print(f'Checkpoints found: {len(checkpoint_paths)}')
print(f'Epoch metrics exists: {METRICS_PATH.exists()}')

Run dir: D:\Курсовая\ich-ct-classification\output\history_3d_residual_cnn_1
Checkpoints found: 10
Epoch metrics exists: True


In [3]:
def tensor_stats(tensors):
    total_sq_sum = 0.0
    total_abs_sum = 0.0
    total_count = 0
    max_abs = 0.0

    for tensor in tensors:
        tensor = tensor.detach().float().cpu()
        total_sq_sum += float(torch.sum(tensor * tensor).item())
        total_abs_sum += float(torch.sum(torch.abs(tensor)).item())
        total_count += tensor.numel()
        max_abs = max(max_abs, float(torch.max(torch.abs(tensor)).item()))

    l2_norm = math.sqrt(total_sq_sum) if total_sq_sum > 0 else 0.0
    mean_abs = total_abs_sum / total_count if total_count > 0 else 0.0
    rms = math.sqrt(total_sq_sum / total_count) if total_count > 0 else 0.0

    return {
        'param_count': total_count,
        'l2_norm': l2_norm,
        'mean_abs': mean_abs,
        'rms': rms,
        'max_abs': max_abs,
    }


def collect_checkpoint_stats(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model_state_dict']
    epoch = int(checkpoint['epoch'])

    all_tensors = list(state_dict.values())
    encoder_tensors = [tensor for name, tensor in state_dict.items() if name.startswith('encoder.')]
    head_tensors = [tensor for name, tensor in state_dict.items() if name.startswith('head.')]

    row = {'epoch': epoch}

    for prefix, stats in [
        ('all', tensor_stats(all_tensors)),
        ('encoder', tensor_stats(encoder_tensors)),
        ('head', tensor_stats(head_tensors)),
    ]:
        for stat_name, value in stats.items():
            row[f'{prefix}_{stat_name}'] = value

    return row

In [4]:
weight_stats_df = pd.DataFrame([collect_checkpoint_stats(path) for path in checkpoint_paths])

if METRICS_PATH.exists():
    epoch_metrics_df = pd.read_csv(METRICS_PATH)
    combined_df = epoch_metrics_df.merge(weight_stats_df, on='epoch', how='left')
else:
    epoch_metrics_df = None
    combined_df = weight_stats_df.copy()

combined_df.round(5)

,epoch,train_loss,val_loss,train_roc_auc,train_pr_auc,val_roc_auc,val_pr_auc,all_param_count,all_l2_norm,all_mean_abs,...,encoder_param_count,encoder_l2_norm,encoder_mean_abs,encoder_rms,encoder_max_abs,head_param_count,head_l2_norm,head_mean_abs,head_rms,head_max_abs
0,1,0.69251,0.69420,0.48259,0.42020,0.60185,0.49746,8316081,29.41091,0.00769,...,8283056,28.66806,0.00759,0.00996,0.24852,33025,6.56841,0.03128,0.03614,0.08801
1,2,0.68727,0.68837,0.50684,0.44222,0.58407,0.52203,8316081,29.46010,0.00770,...,8283056,28.71880,0.00761,0.00998,0.24814,33025,6.56719,0.03127,0.03614,0.08800
2,3,0.68278,0.68356,0.53588,0.49878,0.57370,0.52697,8316081,29.50821,0.00771,...,8283056,28.76834,0.00762,0.01000,0.24834,33025,6.56637,0.03127,0.03613,0.08798
3,4,0.67264,0.65492,0.59359,0.50038,0.65704,0.64505,8316081,29.60668,0.00774,...,8283056,28.86909,0.00764,0.01003,0.24962,33025,6.56747,0.03127,0.03614,0.08815
4,5,0.63729,0.59256,0.67939,0.61596,0.80704,0.76120,8316081,29.76380,0.00777,...,8283056,29.02924,0.00768,0.01009,0.24988,33025,6.57169,0.03129,0.03616,0.08982
5,6,0.59418,0.55457,0.74658,0.68972,0.83667,0.79176,8316081,29.96000,0.00782,...,8283056,29.22831,0.00773,0.01016,0.25163,33025,6.58085,0.03133,0.03621,0.09286
6,7,0.54809,0.52651,0.79487,0.73295,0.81926,0.77841,8316081,30.19005,0.00787,...,8283056,29.46138,0.00778,0.01024,0.25334,33025,6.59289,0.03138,0.03628,0.09591
7,8,0.50196,0.47613,0.83952,0.77389,0.85000,0.78974,8316081,30.41439,0.00793,...,8283056,29.68809,0.00784,0.01032,0.25346,33025,6.60699,0.03144,0.03636,0.10020
8,9,0.42709,0.51826,0.88802,0.86765,0.83148,0.79332,8316081,30.63664,0.00798,...,8283056,29.91169,0.00789,0.01039,0.25287,33025,6.62531,0.03151,0.03646,0.10364
9,10,0.37508,0.53723,0.91321,0.88819,0.81111,0.72466,8316081,30.89626,0.00805,...,8283056,30.17348,0.00795,0.01048,0.25525,33025,6.64381,0.03158,0.03656,0.10677


In [5]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        'Global Weight Norms',
        'Encoder vs Head L2 Norm',
        'Absolute Weight Scale',
        'Validation Metrics'
    )
)

fig.add_trace(go.Scatter(x=combined_df['epoch'], y=combined_df['all_l2_norm'], mode='lines+markers', name='all_l2_norm'), row=1, col=1)
fig.add_trace(go.Scatter(x=combined_df['epoch'], y=combined_df['all_rms'], mode='lines+markers', name='all_rms'), row=1, col=1)

fig.add_trace(go.Scatter(x=combined_df['epoch'], y=combined_df['encoder_l2_norm'], mode='lines+markers', name='encoder_l2_norm'), row=1, col=2)
fig.add_trace(go.Scatter(x=combined_df['epoch'], y=combined_df['head_l2_norm'], mode='lines+markers', name='head_l2_norm'), row=1, col=2)

fig.add_trace(go.Scatter(x=combined_df['epoch'], y=combined_df['all_mean_abs'], mode='lines+markers', name='all_mean_abs'), row=2, col=1)
fig.add_trace(go.Scatter(x=combined_df['epoch'], y=combined_df['all_max_abs'], mode='lines+markers', name='all_max_abs'), row=2, col=1)

if 'val_roc_auc' in combined_df.columns:
    fig.add_trace(go.Scatter(x=combined_df['epoch'], y=combined_df['val_roc_auc'], mode='lines+markers', name='val_roc_auc'), row=2, col=2)
if 'val_pr_auc' in combined_df.columns:
    fig.add_trace(go.Scatter(x=combined_df['epoch'], y=combined_df['val_pr_auc'], mode='lines+markers', name='val_pr_auc'), row=2, col=2)
if 'val_loss' in combined_df.columns:
    fig.add_trace(go.Scatter(x=combined_df['epoch'], y=combined_df['val_loss'], mode='lines+markers', name='val_loss', yaxis='y2'), row=2, col=2)

fig.update_layout(height=900, width=1200, title='Weight Statistics by Epoch', legend_tracegroupgap=120)
fig.update_xaxes(title_text='epoch')
fig.show()

In [6]:
stats_to_plot = [
    'all_l2_norm',
    'all_mean_abs',
    'all_max_abs',
    'encoder_l2_norm',
    'head_l2_norm',
]

long_df = combined_df.melt(id_vars='epoch', value_vars=stats_to_plot, var_name='stat', value_name='value')
px.line(long_df, x='epoch', y='value', color='stat', markers=True, title='Weight Statistics by Epoch')